In [2]:
!nvidia-smi

Tue Jun 30 05:14:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%cd /content
!rm -rf adakv
TOKEN = ""   # ← 粘你的 dafanfan token(别把这个 notebook 分享出去)
!git clone https://{TOKEN}@github.com/liukefan821/adakv.git
%cd /content/adakv

/content
Cloning into 'adakv'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 55 (delta 10), reused 50 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 29.44 KiB | 14.72 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/adakv


In [4]:
!pip install -e ".[gpu]" -q
!pip install -U transformers accelerate -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for adakv (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 115.2 MB/s eta 0:00:00


In [5]:
import transformers, inspect
print("transformers", transformers.__version__)

try:
    from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
    print("ALL_ATTENTION_FUNCTIONS: OK, keys =", list(ALL_ATTENTION_FUNCTIONS.keys()))
except Exception as e:
    print("ALL_ATTENTION_FUNCTIONS: NOT here ->", repr(e))

try:
    from transformers.models.qwen2 import modeling_qwen2 as mq
    print("qwen2 eager sig:", inspect.signature(mq.eager_attention_forward))
except Exception as e:
    print("qwen2 eager fn ->", repr(e))

transformers 5.12.1
ALL_ATTENTION_FUNCTIONS: OK, keys = ['flash_attention_4', 'flash_attention_3', 'flash_attention_2', 'flex_attention', 'sdpa', 'paged|flash_attention_4', 'paged|flash_attention_3', 'paged|flash_attention_2', 'paged|sdpa', 'paged|eager']
qwen2 eager sig: (module: torch.nn.modules.module.Module, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, attention_mask: torch.Tensor | None, scaling: float, dropout: float = 0.0, **kwargs: Unpack[transformers.utils.generic.TransformersKwargs])


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
print("loaded | attn impl:", model.config._attn_implementation)

def chat_generate(prompt, max_new_tokens=40):
    inputs = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(chat_generate("In one sentence, what is a transformer in machine learning?"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded | attn impl: sdpa
A transformer in machine learning refers to a type of neural network architecture designed for tasks involving sequential data, such as natural language processing and speech recognition.


In [7]:
import torch
from adakv.attention import AdaKVCache, plan_selection
from adakv.kernels.block_sparse_decode import HAS_TRITON, block_sparse_decode
from adakv.runtime import _torch_masked_decode
try:
    from transformers import AttentionInterface
except Exception:
    from transformers.modeling_utils import AttentionInterface
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS

_ADAKV_CFG = dict(enabled=True, block_size=16, avg_budget=16, k_min=2, k_max=64,
                  n_sink_blocks=1, n_local_blocks=4, temperature=1.0)
_SDPA = ALL_ATTENTION_FUNCTIONS["sdpa"]

def adakv_attention_forward(module, query, key, value, attention_mask, scaling, dropout=0.0, **kwargs):
    cfg = _ADAKV_CFG
    B, Hq, q_len, _ = query.shape
    # 只拦截单序列单 token 的 decode;其余(prefill/批量)走 sdpa = dense
    if (not cfg["enabled"]) or q_len != 1 or B != 1:
        return _SDPA(module, query, key, value, attention_mask=attention_mask, scaling=scaling, dropout=dropout, **kwargs)
    k = key[0].contiguous()              # [Hkv, S, D]  解码时是完整 KV cache
    v = value[0].contiguous()
    qh = query[0, :, 0, :].contiguous()  # [Hq, D]
    cache = AdaKVCache(block_size=cfg["block_size"]).append_prefill(k, v)
    bt, sl = plan_selection(qh, cache, cfg["avg_budget"], cfg["k_min"], cfg["k_max"],
                            cfg["n_sink_blocks"], cfg["n_local_blocks"], cfg["temperature"])
    if HAS_TRITON and qh.is_cuda:
        out = block_sparse_decode(qh, k, v, bt, sl, cfg["block_size"], sm_scale=scaling)
    else:
        out = _torch_masked_decode(qh, k, v, bt, sl, cfg["block_size"], cache.centroid.shape[1])
    return out.unsqueeze(0).unsqueeze(0).to(query.dtype), None  # [B=1, seq=1, Hq, D]

AttentionInterface.register("adakv", adakv_attention_forward)

def install_adakv(model):
    try:
        model.set_attn_implementation("adakv")
    except Exception:
        model.config._attn_implementation = "adakv"
    return model

def enable_adakv(**cfg):  _ADAKV_CFG.update(cfg); _ADAKV_CFG["enabled"] = True
def disable_adakv():       _ADAKV_CFG["enabled"] = False

print("AdaKV hook registered (sdpa delegation fixed).")

AdaKV hook registered (sdpa delegation fixed).


In [11]:
install_adakv(model)
disable_adakv()
print("FULL  :", chat_generate("Name three primary colors.", max_new_tokens=30))
enable_adakv(block_size=16, avg_budget=8)
print("ADAKV :", chat_generate("Name three primary colors.", max_new_tokens=30))

FULL  : Three primary colors refer to the fundamental colors from which all other colors can be mixed or derived. The three primary colors in the visible spectrum are:

1
ADAKV : Three primary colors refer to the fundamental colors from which all other colors can be mixed or derived. The three primary colors in the visible spectrum are:

1


In [10]:
import importlib, adakv.attention
p = "/content/adakv/adakv/attention.py"
s = open(p).read()
a = "    max_sel = int(kph.clamp_min(sink + local).max().item())"
b = "    max_sel = min(int(kph.clamp_min(sink + local).max().item()), nb)"
c = "        kh = max(int(kph[h]), sink + local)"
d = "        kh = min(max(int(kph[h]), sink + local), nb)"
if a in s:
    open(p, "w").write(s.replace(a, b).replace(c, d))
    print("patched attention.py")
else:
    print("already patched")

importlib.reload(adakv.attention)
from adakv.attention import AdaKVCache, plan_selection      # 重新绑定 hook 用到的全局名
AttentionInterface.register("adakv", adakv_attention_forward)
print("reloaded + re-registered")

patched attention.py
reloaded + re-registered


In [13]:
TOKEN = ""   # ← 粘你的 dafanfan token
!cd /content/adakv && git remote set-url origin https://{TOKEN}@github.com/liukefan821/adakv.git && git push

Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 468 bytes | 468.00 KiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/liukefan821/adakv.git
   6e1618a..d717c40  main -> main


In [14]:
import torch

NEEDLE_VAL = "8675309"
NEEDLE = f"\n\nImportant fact: the secret access code is {NEEDLE_VAL}. Keep it in mind.\n\n"
QUESTION = "\n\nQuestion: What is the secret access code? Reply with only the number."

def build_needle_prompt(n_tokens, depth=0.5):
    filler = ("The history of cartography is long and varied; maps have guided travelers "
              "across deserts and oceans for thousands of years. ")
    reps = n_tokens // max(len(tok(filler).input_ids), 1) + 2
    base_ids = tok(filler * reps).input_ids[:n_tokens]
    insert = int(len(base_ids) * depth)
    needle_ids = tok(NEEDLE, add_special_tokens=False).input_ids
    ids = base_ids[:insert] + needle_ids + base_ids[insert:]   # 把"密码"塞到中间
    return tok.decode(ids)

def run_needle(n_tokens=4000, depth=0.5, max_new_tokens=16):
    user = build_needle_prompt(n_tokens, depth) + QUESTION
    inputs = tok.apply_chat_template([{"role": "user", "content": user}],
                                     add_generation_prompt=True, return_tensors="pt",
                                     return_dict=True).to("cuda")
    ctx = inputs["input_ids"].shape[1]
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    ans = tok.decode(out[0, ctx:], skip_special_tokens=True)
    return (NEEDLE_VAL in ans), ans, ctx

print("needle helpers ready")

needle helpers ready


In [15]:
N = 4000   # 上下文长度(token);密码埋在正中(depth=0.5)

disable_adakv()
ok_full, ans_full, ctx = run_needle(N)
print(f"context tokens ≈ {ctx}\n")
print(f"FULL               retrieved={ok_full}   ans={ans_full!r}")

for bud in [8, 16, 32]:
    enable_adakv(block_size=16, avg_budget=bud)
    ok, ans, _ = run_needle(N)
    kv = bud * 16
    print(f"ADAKV budget={bud:>2}  (~{kv}/{ctx} tok ≈ 1/{max(ctx//kv,1)})  retrieved={ok}   ans={ans!r}")

context tokens ≈ 3955

FULL               retrieved=True   ans='8675309'
ADAKV budget= 8  (~128/3955 tok ≈ 1/30)  retrieved=True   ans='8675309'
ADAKV budget=16  (~256/3955 tok ≈ 1/15)  retrieved=True   ans='8675309'
ADAKV budget=32  (~512/3955 tok ≈ 1/7)  retrieved=True   ans='8675309'


In [16]:
%%writefile adakv/patch.py
"""Plug AdaKV into a HuggingFace model via the attention interface (transformers >= 5.x).

Usage:
    from adakv.patch import install_adakv, enable_adakv, disable_adakv
    install_adakv(model)
    enable_adakv(block_size=16, avg_budget=16)
    # ... model.generate(...) ...
    disable_adakv()

Only single-sequence (batch 1), single-token decode steps are intercepted; prefill
and batched calls delegate to sdpa (dense). Centroids are recomputed from the full
KV each decode step (correctness-first; incremental update is a separate optimisation).
Validated: needle retrieved at ~1/30 KV budget on Qwen2.5-1.5B-Instruct.
"""
from __future__ import annotations

import torch  # noqa: F401

from .attention import AdaKVCache, plan_selection
from .kernels.block_sparse_decode import HAS_TRITON, block_sparse_decode
from .runtime import _torch_masked_decode

try:
    from transformers import AttentionInterface
except Exception:
    from transformers.modeling_utils import AttentionInterface
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS

_ADAKV_CFG = dict(enabled=True, block_size=16, avg_budget=16, k_min=2, k_max=64,
                  n_sink_blocks=1, n_local_blocks=4, temperature=1.0)
_SDPA = ALL_ATTENTION_FUNCTIONS["sdpa"]


def adakv_attention_forward(module, query, key, value, attention_mask, scaling, dropout=0.0, **kwargs):
    cfg = _ADAKV_CFG
    B, Hq, q_len, _ = query.shape
    if (not cfg["enabled"]) or q_len != 1 or B != 1:
        return _SDPA(module, query, key, value, attention_mask=attention_mask,
                     scaling=scaling, dropout=dropout, **kwargs)
    k = key[0].contiguous()
    v = value[0].contiguous()
    qh = query[0, :, 0, :].contiguous()
    cache = AdaKVCache(block_size=cfg["block_size"]).append_prefill(k, v)
    bt, sl = plan_selection(qh, cache, cfg["avg_budget"], cfg["k_min"], cfg["k_max"],
                            cfg["n_sink_blocks"], cfg["n_local_blocks"], cfg["temperature"])
    if HAS_TRITON and qh.is_cuda:
        out = block_sparse_decode(qh, k, v, bt, sl, cfg["block_size"], sm_scale=scaling)
    else:
        out = _torch_masked_decode(qh, k, v, bt, sl, cfg["block_size"], cache.centroid.shape[1])
    return out.unsqueeze(0).unsqueeze(0).to(query.dtype), None


AttentionInterface.register("adakv", adakv_attention_forward)


def install_adakv(model):
    try:
        model.set_attn_implementation("adakv")
    except Exception:
        model.config._attn_implementation = "adakv"
    return model


def enable_adakv(**cfg):
    _ADAKV_CFG.update(cfg)
    _ADAKV_CFG["enabled"] = True


def disable_adakv():
    _ADAKV_CFG["enabled"] = False

Overwriting adakv/patch.py


In [17]:
from adakv.patch import install_adakv, enable_adakv, disable_adakv
install_adakv(model)
disable_adakv();          print("FULL ", run_needle(4000)[:2])
enable_adakv(avg_budget=8); print("ADAKV", run_needle(4000)[:2])

FULL  (True, '8675309')
ADAKV (True, '8675309')


In [18]:
!cd /content/adakv && git add -A && git commit -m "Add adakv/patch.py: HF attention-interface hook (needle retrieved at 1/30 KV on Qwen2.5-1.5B)" && git push

[main dbbd6bc] Add adakv/patch.py: HF attention-interface hook (needle retrieved at 1/30 KV on Qwen2.5-1.5B)
 1 file changed, 70 insertions(+), 39 deletions(-)
 rewrite adakv/patch.py (95%)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 1.48 KiB | 1.48 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/liukefan821/adakv.git
   d717c40..dbbd6bc  main -> main


In [19]:
print("depth sweep @ N=4000, ADAKV budget=8 (~1/30 KV)\n")
for d in [0.1, 0.25, 0.5, 0.75, 0.9]:
    disable_adakv();           ok_f = run_needle(4000, depth=d)[0]
    enable_adakv(avg_budget=8); ok_a = run_needle(4000, depth=d)[0]
    print(f"depth={d:>4}:  FULL={ok_f}   ADAKV(1/30)={ok_a}")

depth sweep @ N=4000, ADAKV budget=8 (~1/30 KV)

depth= 0.1:  FULL=True   ADAKV(1/30)=True
depth=0.25:  FULL=True   ADAKV(1/30)=True
depth= 0.5:  FULL=True   ADAKV(1/30)=True
depth=0.75:  FULL=True   ADAKV(1/30)=True
depth= 0.9:  FULL=True   ADAKV(1/30)=True
